# YOLO Aircraft Detection — Training Notebook

End-to-end Colab workflow for training a single-class fixed-wing aircraft detector:

1. Mount Google Drive and install dependencies
2. Extract the dataset ZIP archive
3. *(Optional)* Create a smaller random subset for faster experiments
4. Split images into train/validation sets and generate `data.yaml`
5. *(Optional)* Run hyperparameter tuning with `model.tune()`
6. Train YOLO11 with custom augmentation settings
7. Free GPU memory and optionally restart the runtime

Edit the **Configuration** cell below before running.

## Configuration

Update all `path/to/...` placeholders and training parameters before running.

In [ ]:
from pathlib import Path

# --- Google Drive paths (update before running) ---
DRIVE_BASE = Path("/content/drive/MyDrive/path/to/project")
DATASET_ZIP = DRIVE_BASE / "path/to/dataset.zip"
TRAINED_MODELS_DIR = DRIVE_BASE / "path/to/runs"
TUNE_PROJECT_DIR = DRIVE_BASE / "path/to/tune"

# --- Local workspace paths ---
EXTRACT_DIR = Path("/content/dataset")
SUBSET_DIR = Path("/content/subset")
SPLIT_DIR = Path("/content/dataset_split")

# --- Dataset options ---
USE_SUBSET = False       # True: train on a random subset instead of the full dataset
NUM_SUBSET_FILES = 2000
VAL_RATIO = 0.20
CLASS_NAMES = ["plane"]

# --- Training ---
BASE_MODEL = "yolo11n.pt"
RESUME_CHECKPOINT = TRAINED_MODELS_DIR / "run_name/weights/last.pt"
TRAIN_NAME = "plane_detector"
EPOCHS = 400
IMAGE_SIZE = 640
BATCH_SIZE = 0.95  # adjust for your GPU (e.g. ~149 for A100 40GB, ~16 for T4) (0.95 for %95 gpu ram allocation)
SAVE_PERIOD = 25

# Augmentation (used when tuning is disabled)
AUGMENTATION = {
    "degrees": 45,
    "translate": 0.20,
    "fliplr": 0.4,
    "scale": 0.1,
}

# --- Hyperparameter tuning (optional) ---
RUN_TUNING = False
TUNE_NAME = "tune_run"
TUNE_EPOCHS = 20
TUNE_ITERATIONS = 200
TUNE_SEARCH_SPACE = {
    "degrees": (10.0, 45.0),
    "hsv_v": (0.3, 0.5),
    "fliplr": (0.3, 0.8),
    "translate": (0.3, 0.8),
    "scale": (0.1, 0.5),
    "cutmix": (0.01, 0.5),
    "mixup": (0.01, 0.3),
    "mosaic": (0.1, 1.0),
    "perspective": (0.0001, 0.001),
}

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Install dependencies

In [ ]:
%pip install -q ultralytics

## 3. Extract dataset

Unpacks the ZIP archive into `EXTRACT_DIR`. Expected layout after extraction:

```
dataset/
├── images/
└── labels/
```

In [ ]:
import zipfile

with zipfile.ZipFile(DATASET_ZIP, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR.parent)

print(f"Dataset extracted to: {EXTRACT_DIR}")

## 4. Create subset *(optional)*

Randomly samples `NUM_SUBSET_FILES` image/label pairs into `SUBSET_DIR`.
Set `USE_SUBSET = True` in the configuration cell to use this smaller dataset for splitting.

In [ ]:
import os
import random
import shutil

from tqdm import tqdm


def create_subset(source_images, source_labels, output_dir, num_files):
    """Copy a random subset of images and matching label files."""
    output_images = output_dir / "images"
    output_labels = output_dir / "labels"
    output_images.mkdir(parents=True, exist_ok=True)
    output_labels.mkdir(parents=True, exist_ok=True)

    image_files = [
        f for f in os.listdir(source_images)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ]

    if len(image_files) < num_files:
        print(
            f"Warning: requested {num_files} files but only {len(image_files)} available. "
            "Using all images."
        )
        selected = image_files
    else:
        selected = random.sample(image_files, num_files)

    for image_name in tqdm(selected, desc="Copying subset"):
        shutil.copy2(source_images / image_name, output_images / image_name)

        label_name = f"{Path(image_name).stem}.txt"
        label_path = source_labels / label_name
        if label_path.exists():
            shutil.copy2(label_path, output_labels / label_name)
        else:
            print(f"Warning: label not found for '{image_name}', skipped.")

    print(f"Subset created at '{output_dir}' ({len(selected)} images).")


if USE_SUBSET:
    create_subset(
        EXTRACT_DIR / "images",
        EXTRACT_DIR / "labels",
        SUBSET_DIR,
        NUM_SUBSET_FILES,
    )
else:
    print("Skipping subset creation (USE_SUBSET=False).")

## 5. Train/validation split and `data.yaml`

Splits the dataset into `train/` and `val/` folders and writes the Ultralytics config file.

In [ ]:
import glob
import os
import random
import shutil

import yaml

source_dir = SUBSET_DIR if USE_SUBSET else EXTRACT_DIR
images_dir = source_dir / "images"
labels_dir = source_dir / "labels"

train_images_dir = SPLIT_DIR / "train/images"
train_labels_dir = SPLIT_DIR / "train/labels"
val_images_dir = SPLIT_DIR / "val/images"
val_labels_dir = SPLIT_DIR / "val/labels"

for directory in [train_images_dir, train_labels_dir, val_images_dir, val_labels_dir]:
    directory.mkdir(parents=True, exist_ok=True)

image_files = []
for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
    image_files.extend(glob.glob(str(images_dir / ext)))

print(f"Found {len(image_files)} images in '{source_dir}'.")

random.shuffle(image_files)

val_count = int(len(image_files) * VAL_RATIO)
val_files = image_files[:val_count]
train_files = image_files[val_count:]


def copy_split(file_list, target_images, target_labels):
    for image_path in file_list:
        filename = os.path.basename(image_path)
        stem = Path(filename).stem
        label_path = labels_dir / f"{stem}.txt"

        shutil.copy2(image_path, target_images / filename)
        if label_path.exists():
            shutil.copy2(label_path, target_labels / f"{stem}.txt")
        else:
            print(f"Warning: label not found for '{filename}'.")


copy_split(train_files, train_images_dir, train_labels_dir)
copy_split(val_files, val_images_dir, val_labels_dir)

print(f"Train: {len(train_files)} | Val: {len(val_files)}")

data_yaml = {
    "path": str(SPLIT_DIR),
    "train": "train/images",
    "val": "val/images",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}

yaml_path = SPLIT_DIR / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f"data.yaml written to: {yaml_path}")

## 6. Hyperparameter tuning *(optional)*

Runs Ultralytics `model.tune()` to search augmentation hyperparameters.
Set `RUN_TUNING = True` in the configuration cell to enable.

In [ ]:
from ultralytics import YOLO

if RUN_TUNING:
    tune_model = YOLO(BASE_MODEL)
    tune_model.tune(
        data=str(SPLIT_DIR / "data.yaml"),
        space=TUNE_SEARCH_SPACE,
        epochs=TUNE_EPOCHS,
        iterations=TUNE_ITERATIONS,
        optimizer="AdamW",
        cache=True,
        plots=True,
        save=True,
        val=True,
        batch=0.95,
        project=str(TUNE_PROJECT_DIR),
        name=TUNE_NAME,
    )
else:
    print("Skipping hyperparameter tuning (RUN_TUNING=False).")

## 7. Train model

Fine-tunes YOLO on the plane detection dataset. Set `resume=True` to continue
from `RESUME_CHECKPOINT` if a previous run was interrupted.

In [ ]:
from ultralytics import YOLO

model = YOLO(str(RESUME_CHECKPOINT))

model.train(
    data=str(SPLIT_DIR / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    cache=True,
    single_cls=True,
    plots=True,
    save_period=SAVE_PERIOD,
    resume=True,
    project=str(TRAINED_MODELS_DIR),
    name=TRAIN_NAME,
    **AUGMENTATION,
)

## 8. Free GPU memory

In [ ]:
import gc

import torch

try:
    del model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

print("GPU and RAM cleared.")
!nvidia-smi

## 9. Restart runtime *(optional)*

Use this if Colab still holds stale GPU memory after cleanup.

In [ ]:
import os

print("Restarting runtime...")
os.kill(os.getpid(), 9)